# Trust Calibration and Reliance Gap Lab


## Purpose

This lab introduces trust calibration.

Learning goals:

- Simulate model confidence and correctness.
- Simulate user reliance.
- Compute overreliance, underreliance, and reliance gaps.
- Interpret trust as calibrated reliance.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 1500

data = pd.DataFrame({
    "case_id": [f"C-{i:04d}" for i in range(1, n + 1)],
    "model_confidence": rng.beta(5, 2, size=n),
    "user_expertise": rng.choice(["novice", "intermediate", "expert"], size=n, p=[0.30, 0.45, 0.25]),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
})

data["model_correct"] = rng.binomial(
    1,
    p=np.clip(0.20 + 0.75 * data["model_confidence"], 0, 1),
    size=n,
)

data["user_relied_on_ai"] = rng.binomial(
    1,
    p=np.clip(0.15 + 0.65 * data["model_confidence"], 0.02, 0.98),
    size=n,
)

data["overreliance"] = ((data["user_relied_on_ai"] == 1) & (data["model_correct"] == 0)).astype(int)
data["underreliance"] = ((data["user_relied_on_ai"] == 0) & (data["model_correct"] == 1)).astype(int)
data["reliance_gap"] = abs(data["user_relied_on_ai"] - data["model_correct"])

summary = (
    data
    .groupby(["user_expertise", "risk_level"])
    .agg(
        cases=("case_id", "count"),
        accuracy=("model_correct", "mean"),
        reliance_rate=("user_relied_on_ai", "mean"),
        overreliance_rate=("overreliance", "mean"),
        underreliance_rate=("underreliance", "mean"),
        mean_reliance_gap=("reliance_gap", "mean"),
    )
    .reset_index()
)

summary

## Interpretation

Trust is best treated as calibrated reliance rather than general confidence in the system.